## Q4: Data statistics




In [ ]:
from pathlib import Path
import sys
from transformers import T5TokenizerFast
from IPython.display import Markdown, display


data_dir = next((path.resolve() for path in
                 (Path("data"), Path("part2/data"), Path("nyu-dsga1011-a4/part2/data"))
                 if (path / "train.nl").is_file() and (path / "dev.sql").is_file()), None)


sys.path.insert(0, str(data_dir.parent))
from load_data import load_lines, data_statistics, T5Dataset

tokenizer = T5TokenizerFast.from_pretrained("google-t5/t5-small")

def print_table(title, rows, include_count=True):
    labels = ["Number of examples", "Mean sentence length", "Mean SQL query length",
              "Vocabulary size (natural language)", "Vocabulary size (SQL)"]
    table = ["| Statistics Name | Train | Dev |", "|---|---:|---:|"]
    for i, label in enumerate(labels):
        if i == 0 and not include_count:
            continue
        def format_value(value):
            return f"{value:.2f}" if isinstance(value, float) else str(value)
        table.append(f"| {label} | {format_value(rows['train'][i])} | {format_value(rows['dev'][i])} |")
    display(Markdown(f"**{title}**\n\n" + "\n".join(table)))

# Preserve the original text apart from removing file line endings.
raw = {split: (load_lines(data_dir / f"{split}.nl", strip=False),
               load_lines(data_dir / f"{split}.sql", strip=False))
       for split in ("train", "dev")}
raw_statistics = {split: data_statistics(pair[0], pair[1], tokenizer)
                  for split, pair in raw.items()}
print_table("Table 1: Before preprocessing", raw_statistics)


/opt/anaconda3/envs/dsga1011-hw4-part1/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[nltk_data] Downloading package punkt to /Users/ivysim/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
/opt/anaconda3/envs/dsga1011-hw4-part1/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


**Table 1: Before preprocessing**

| Statistics Name | Train | Dev |
|---|---:|---:|
| Number of examples | 4225 | 466 |
| Mean sentence length | 17.10 | 17.07 |
| Mean SQL query length | 216.37 | 210.05 |
| Vocabulary size (natural language) | 791 | 465 |
| Vocabulary size (SQL) | 555 | 395 |

### Preprocessing

- **Clean up line breaks:** replace actual `\n` and `\r` characters inside English questions and SQL queries with spaces.
- **Remove surrounding whitespace:** strip spaces at the beginning and end of each question and query. Keep the original capitalization and internal spacing.
- **Check SQL syntax and database references:** use SQLite `EXPLAIN` with a read-only database connection. This checks whether SQLite can compile each query without executing it. Flag failed queries for review; keep the examples rather than deleting or automatically rewriting them.
- **Format the database schema as DDL:** save a separate file of `CREATE TABLE` statements containing table names, column names, data types, and the primary/foreign key constraints actually declared in the database. This database declares four primary keys and no foreign keys. Save the JSON schema's relationship links separately as hints. The DDL is not appended to the English questions.
- **Preserve the data pairs and order:** keep every English question aligned with its SQL query. No training or development examples are removed. The test split has English questions only, so it receives text cleanup without SQL-target validation.


In [3]:
processed = {split: T5Dataset(data_dir, split) for split in ("train", "dev")}
print_table("Table 2: After preprocessing",
            {split: data_statistics(dataset.input, dataset.target, tokenizer)
             for split, dataset in processed.items()},
            include_count=True)


/Users/ivysim/code/Ivy_Sim_1011/nyu-dsga1011-a4/nyu-dsga1011-a4/part2/load_data.py:37: UserWarning: train: 1 SQL queries failed SQLite validation; examples retained. Inspect dataset.sql_validation['errors'].
  self.input, self.target = self.process_data(self.data_folder, self.split, self.tokenizer)


**Table 2: After preprocessing**

| Statistics Name | Train | Dev |
|---|---:|---:|
| Number of examples | 4225 | 466 |
| Mean sentence length | 17.10 | 17.07 |
| Mean SQL query length | 216.37 | 210.05 |
| Vocabulary size (natural language) | 791 | 465 |
| Vocabulary size (SQL) | 555 | 395 |

The statistics stay the same because the cleanup does not change these T5 token counts, SQL validation does not remove examples, and the separate DDL file is not part of the model input. Lengths and vocabulary sizes exclude padding and added special tokens; vocabulary size counts distinct T5 token IDs within each split.


In [2]:
from load_data import preprocess_data_files

preprocessing_dir = data_dir.parents[2] / "output/part2/preprocessing"
preprocessing_report = preprocess_data_files(data_dir, preprocessing_dir)
for split in ("train", "dev"):
    validation = preprocessing_report["splits"][split]["sql_validation"]
    print(f"{split}: {validation['valid']}/{validation['total']} SQL queries pass SQLite validation")
    for error in validation["errors"]:
        print(f"  Line {error['line']}: {error['error']} (retained for review)")
print("DDL:", preprocessing_dir / "flight_database_ddl.sql")
print("Validation report:", preprocessing_dir / "preprocessing_report.json")


train: 4224/4225 SQL queries pass SQLite validation
  Line 1349: near "ALL": syntax error (retained for review)
dev: 466/466 SQL queries pass SQLite validation
DDL: /Users/ivysim/code/Ivy_Sim_1011/nyu-dsga1011-a4/output/part2/preprocessing/flight_database_ddl.sql
Validation report: /Users/ivysim/code/Ivy_Sim_1011/nyu-dsga1011-a4/output/part2/preprocessing/preprocessing_report.json


## Q5: Fine-tuning hyperparameter experiments

| Design choice | Description |
|---|---|
| Data processing | I replaced actual line breaks with spaces and removed surrounding whitespace, while keeping capitalization and the question–SQL pairs unchanged. I checked the training and development SQL using read-only SQLite `EXPLAIN`; flagged queries were kept rather than removed or rewritten. I also exported the database schema as a separate DDL file with its declared constraints. The schema was not appended to the model input. |
| Tokenization | I used `T5TokenizerFast` from `google-t5/t5-small` for both the English encoder inputs and SQL decoder targets. This uses the pretrained T5 tokenizer vocabulary, rather than a custom tokenizer. Both sequences include the end-of-sequence token. During training, decoder inputs are the SQL tokens shifted right, starting with T5’s padding token. Sequences are padded to the longest sequence in each batch, encoder padding is masked, and padding targets are excluded from the loss. I did not truncate the training sequences. |
| Architecture | I fine-tuned the entire pretrained `google-t5/t5-small` model: the encoder, decoder, attention layers, feed-forward layers, shared embeddings, and output projection. No layers were frozen, and I did not change the number of layers or hidden dimensions. |
| Hyperparameters | I tested learning rates `5e-5`, `1e-4`, and `3e-4`, batch sizes `8` and `16`, and weight decay `0` and `0.01` in 12 three-epoch runs. These used AdamW, linear learning-rate decay, no warmup, seed `42`, and early-stopping patience of 2 epochs based on full-development loss. Run 9 (`3e-4`, batch size `8`, weight decay `0`) was selected using the fixed 32-example development comparison. I then continued its weights for 27 more epochs with a fresh optimizer and scheduler, learning rate `1e-4`, batch size `8`, and weight decay `0`. For continuation, I selected checkpoints by full-development Record F1, evaluated every 3 additional epochs, with patience of 3 F1 evaluations and a maximum of 30 total epochs. The run reached 30 epochs, and epoch 30 was the best checkpoint. Final SQL generation used `num_beams=4`, `length_penalty=0.6`, and `max_new_tokens=512`, chosen through development experiments. |

### Final selected-model results

| Model | Dev Record F1 (%) | Dev Record EM (%) | Dev SQL EM (%) |
|---|---:|---:|---:|
| Three-epoch run 9, 4 beams / length penalty 0.6 | 57.50 | 49.14 | 1.93 |
| Continued run 9, selected at 30 total epochs | 81.15 | 79.83 | 3.00 |



## Q6: Results and error analysis

### Table 4: Fine-tuned T5 results

| System | Query EM (%) | F1 score (%) |
|---|---:|---:|
| **Dev Results** | | |
| T5 fine-tuned — full model | 3.00 | 81.15 |
| **Test Results** | | |
| T5 fine-tuned — full model | 0.00 | 75.53 |

These development results use all 466 examples and the selected 30-epoch checkpoint. The model can produce different SQL that returns the correct records, so Query EM is much lower than Record F1. The test scores are from the official evaluation of `t5_ft_experiment_test.pkl` from gradescope. 




### Table 5: Qualitative error analysis

I compared the generated SQL with the reference SQL and database records on all **466 development examples**. The examples below use zero-based `example_index` values from `dev_comparison.csv`.

| Error Type | Example of Error | Error Description | Statistics (COUNT/TOTAL) |
|---|---|---|---|
| Malformed SQL | Example 19: “flights … in the morning before 8.” Reference: `flight_1.departure_time < 800`; generated: `flight_1.departure_time  800`. | The comparison operator is missing, so SQLite cannot execute the query. This category includes syntax errors and incomplete queries. Generating long SQL token by token can leave out an operator or produce unmatched parentheses. | **37/466** |
| Invalid schema reference | Example 90: “what aircraft is co 1209.” Generated: `aircraft_1.flight_number = 1209`; SQLite reports `no such column: aircraft_1.flight_number`. | The model uses a column on the wrong table, refers to an unavailable alias, or produces an ambiguous column reference. It may associate “aircraft” with the question without correctly linking the aircraft and flight tables. | **13/466** |
| Wrong output columns | Example 97 asks what transportation is available and “how much does it cost.” Reference selects `transport_type, ground_fare`; generated SQL selects only `transport_type`. | The query runs but returns the wrong fields or leaves out part of the requested answer. Here, the model answers the transportation part and misses the cost. This category counts executable queries with incorrect records and a different normalized outer `SELECT` clause. | **13/466** |
| Wrong query conditions or structure | Example 22: “arriving … at about 4 o’clock.” Reference: `arrival_time >= 1530 AND arrival_time <= 1630`; generated: `arrival_time >= 1630 AND arrival_time = 1630`. | The query runs and selects the same output fields, but its conditions, joins, or subqueries return incorrect records. Here, the model turns an approximate time window into a single time. Other examples confuse airline codes, omit a flight number, or miss a return trip. Translating these details into the right SQL conditions is harder than copying the main flight-query pattern. | **40/466** |

**How I counted:** each query receives at most one category. I first classify execution errors using the saved SQLite error message. For queries that execute but have incorrect records, I compare the outer `SELECT` clauses, ignoring whitespace and case: a different clause goes in “wrong output columns”; the remaining queries go in “wrong query conditions or structure.” This is a practical grouping for analysis, rather than a complete SQL-equivalence check. A query can have additional mistakes beyond its assigned category. The row-level assignments and SQL pairs are saved in `results/dev_error_types.csv`.

There are **103/466** queries with an execution error or incorrect records. This is higher than the **94/466** record mismatches because **9 execution errors** happen to match an empty reference record set in the saved record comparison. I still count these as errors because the SQL failed to execute. I do not count formatting-only SQL differences as semantic errors when the returned records are correct. The possible reasons above are interpretations of the observed examples, not measured causes.

